In [99]:
%pip install numpy matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\USUARIO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Manipulador 5-GDL — Matriz DH (visualización 3D simple)

Tabla DH (convención modificada de Craig: $T_i^{i-1} = Rot_x(\alpha_{i-1}) \cdot Trans_x(a_{i-1}) \cdot Rot_z(\theta_i) \cdot Trans_z(d_i)$):

| i | $\alpha_{i-1}$ | $a_{i-1}$ | $\theta_i$ | $d_i$ | $\sigma_i$ |
|---|---|---|---|---|---|
| 1 | $\pi/2$  | 0    | $\pi/2$ (fijo) | $d_1$ (variable) | 1 (prismática) |
| 2 | 0        | 0.33 | $\theta_2$ (variable) | 0    | 0 (revoluta) |
| 3 | 0        | 0.4  | $\theta_3$ (variable) | 0    | 0 (revoluta) |
| 4 | $\pi/2$  | 0    | $\theta_4$ (variable) | 0.16 | 0 (revoluta) |
| 5 | $-\pi/2$ | 0    | $\theta_5$ (variable) | 0    | 0 (revoluta) |

$\sigma_i = 1$ indica junta prismática (la variable articular es $d_i$); $\sigma_i = 0$ indica junta revoluta (la variable articular es $\theta_i$).

In [100]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

%matplotlib tk
# Abre cada gráfico en una ventana aparte: arrastra con el mouse para rotar.

## Matriz de transformación homogénea DH (una fila de la tabla)

In [101]:
def dh(alpha, a, theta, d):
    """Transformación homogénea DH modificada (Craig) para una fila de la tabla."""
    ca, sa = np.cos(alpha), np.sin(alpha)
    ct, st = np.cos(theta), np.sin(theta)
    return np.array([
        [ct,     -st,      0,     a],
        [st*ca,   ct*ca,  -sa,   -sa*d],
        [st*sa,   ct*sa,   ca,    ca*d],
        [0,       0,       0,     1]
    ])

## Tabla DH (mismos nombres/columnas de la imagen) y tipo de articulación

Cada fila usa las claves `i`, `alpha`, `a`, `theta`, `d`, `sigma` — igual que las columnas $i, \alpha_{i-1}, a_{i-1}, \theta_i, d_i, \sigma_i$ de la tabla. `sigma` nos dice si la articulación es **prismática** ($\sigma=1$) o **bisagra** ($\sigma=0$).

In [102]:
def tabla_dh(q):
    """Tabla DH del manipulador, con los mismos nombres de columna que la imagen."""
    d1, th2, th3, th4, th5 = q
    return [
        {"i": 1, "alpha": np.pi/2,  "a": 0.0,  "theta": np.pi/2, "d": d1,   "sigma": 1},
        {"i": 2, "alpha": 0.0,      "a": 0.33, "theta": th2,     "d": 0.0,  "sigma": 0},
        {"i": 3, "alpha": 0.0,      "a": 0.4,  "theta": th3,     "d": 0.0,  "sigma": 0},
        {"i": 4, "alpha": np.pi/2,  "a": 0.0,  "theta": th4,     "d": 0.16, "sigma": 0},
        {"i": 5, "alpha": -np.pi/2, "a": 0.0,  "theta": th5,     "d": 0.0,  "sigma": 0},
    ]


def tipo_articulacion(sigma):
    """sigma = 1 -> prismática; sigma = 0 -> bisagra."""
    return "prismática" if sigma == 1 else "bisagra"

In [103]:
for fila in tabla_dh([0.25, 0, 0, 0, 0]):
    tipo = tipo_articulacion(fila["sigma"])
    print(f"i={fila['i']}  alpha={fila['alpha']:+.4f} rad  "
          f"a={fila['a']:.3f}  theta={fila['theta']:+.4f} rad  "
          f"d={fila['d']:.3f}  sigma={fila['sigma']}  ->  articulación {tipo}")

i=1  alpha=+1.5708 rad  a=0.000  theta=+1.5708 rad  d=0.250  sigma=1  ->  articulación prismática
i=2  alpha=+0.0000 rad  a=0.330  theta=+0.0000 rad  d=0.000  sigma=0  ->  articulación bisagra
i=3  alpha=+0.0000 rad  a=0.400  theta=+0.0000 rad  d=0.000  sigma=0  ->  articulación bisagra
i=4  alpha=+1.5708 rad  a=0.000  theta=+0.0000 rad  d=0.160  sigma=0  ->  articulación bisagra
i=5  alpha=-1.5708 rad  a=0.000  theta=+0.0000 rad  d=0.000  sigma=0  ->  articulación bisagra


## Cinemática directa del manipulador de 5 GDL

`q = [d1, theta2, theta3, theta4, theta5]` son las 5 variables articulares (d1 en metros, los ángulos en radianes).

In [104]:
def forward_kinematics(q):
    T = np.eye(4)
    frames = [T.copy()]  # frame 0 (base)
    for fila in tabla_dh(q):
        T = T @ dh(fila["alpha"], fila["a"], fila["theta"], fila["d"])
        frames.append(T.copy())
    return frames

## Graficado 3D simple

Dibuja los eslabones como una línea que une el origen de cada frame, y los ejes X (verde) / Z (rojo) de cada frame.

In [105]:
def plot_robot(q, axis_len=0.05):
    frames = forward_kinematics(q)
    puntos = np.array([F[:3, 3] for F in frames])

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')

    # eslabones
    ax.plot(puntos[:, 0], puntos[:, 1], puntos[:, 2], '-o', color='black', linewidth=2, markersize=6)

    # ejes locales de cada frame
    for F in frames:
        origen = F[:3, 3]
        x_axis = F[:3, 0]
        z_axis = F[:3, 2]
        ax.quiver(*origen, *x_axis, length=axis_len, color='green')
        ax.quiver(*origen, *z_axis, length=axis_len, color='red')

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('Manipulador 5-GDL (DH)')

    rango = max(puntos.max() - puntos.min(), 0.1)
    centro = puntos.mean(axis=0)
    ax.set_xlim(centro[0] - rango, centro[0] + rango)
    ax.set_ylim(centro[1] - rango, centro[1] + rango)
    ax.set_zlim(centro[2] - rango, centro[2] + rango)

    plt.show()

## Ejemplo 1: pose del dibujo del enunciado (riel → poste → brazo → caída → gripper)

En el diagrama del taller (el robot con el riel de 8 cm, el poste de 25 cm, el brazo de
40 cm y la caída de 16+14 cm hasta el gripper) los tres primeros eslabones forman una
escuadra: el poste sube, el brazo sale horizontal, y luego cae hacia la muñeca. Con
nuestra tabla DH eso se logra con $\theta_2=-90°$ (no con $\theta_2=0$, que en realidad
deja el poste y el brazo apilados en la misma dirección vertical). $d_1=0.25$ m es la
posición del riel que aparece dibujada, y $\theta_3=\theta_4=\theta_5=0$.

In [ ]:
q0 = [0.25, -np.pi/2, 0, 0, 0]  # d1 = 0.25 m, theta2 = -90 grados (ver explicación arriba)
plot_robot(q0)

## Ejemplo 2: cambiando las variables articulares

In [107]:
q1 = [0.25, np.pi/4, -np.pi/6, np.pi/3, 0]
plot_robot(q1)

---

# Parte 2: el mismo robot usando `roboticstoolbox-python`

Ahora repetimos exactamente el mismo manipulador, pero usando la librería `roboticstoolbox-python` en vez de las matrices DH escritas a mano. Como nuestra tabla usa la convención **DH modificada (Craig)** — columnas $\alpha_{i-1}, a_{i-1}, \theta_i, d_i$ — usamos las clases `RevoluteMDH` / `PrismaticMDH` (el sufijo `MDH` es justamente por "Modified DH").

In [108]:
%pip install roboticstoolbox-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\USUARIO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [109]:
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH

robot = DHRobot([
    PrismaticMDH(theta=np.pi/2, a=0,    alpha=np.pi/2),   # junta 1: prismática
    RevoluteMDH(d=0,    a=0.33, alpha=0),                  # junta 2: bisagra
    RevoluteMDH(d=0,    a=0.4,  alpha=0),                  # junta 3: bisagra
    RevoluteMDH(d=0.16, a=0,    alpha=np.pi/2),            # junta 4: bisagra
    RevoluteMDH(d=0,    a=0,    alpha=-np.pi/2),           # junta 5: bisagra
], name="Manipulador 5GDL")

print(robot)  # "PRRRR" = 1 prismática + 4 bisagra

DHRobot: Manipulador 5GDL, 5 joints (PRRRR), dynamics, modified DH parameters
┌──────┬────────┬───────┬──────┐
│ aⱼ₋₁ │  ⍺ⱼ₋₁  │  θⱼ   │  dⱼ  │
├──────┼────────┼───────┼──────┤
│    0 │  90.0° │ 90.0° │   q1 │
│ 0.33 │   0.0° │    q2 │    0 │
│  0.4 │   0.0° │    q3 │    0 │
│    0 │  90.0° │    q4 │ 0.16 │
│    0 │ -90.0° │    q5 │    0 │
└──────┴────────┴───────┴──────┘



## Cinemática directa y graficado con la librería

`fkine(q)` calcula la matriz de transformación homogénea del efector final (equivalente a nuestro `forward_kinematics(q)[-1]` de la Parte 1). `robot.plot(q)` grafica el robot en 3D automáticamente, sin necesidad de escribir código de graficado a mano.

In [ ]:
q0 = [0.25, -np.pi/2, 0, 0, 0]  # mismos valores que en la Parte 1 (pose del dibujo)

T = robot.fkine(q0)
print(T)

plt.rcParams['figure.figsize'] = (8, 8)
robot.plot(q0, block=True, jointaxes=True, eeframe=True, shadow=True)

## Ejemplo con otra pose (mismos ángulos que en la Parte 1)

In [ ]:
q1 = [0.25, np.pi/4, -np.pi/6, np.pi/3, 0]

plt.rcParams['figure.figsize'] = (8, 8)
robot.plot(q1, block=True, jointaxes=True, eeframe=True, shadow=True)

---

# Parte 3: Animación de movimiento + gripper abierto (versión "sólida")

En el caso de estudio de referencia el movimiento se muestra articulación por
articulación (primero $d_1$, luego $\theta_2$, luego $\theta_3$, luego $\theta_4$ y por
último $\theta_5$), redibujando el robot cuadro a cuadro con `clear_output` +
`display(fig)`. Esa técnica funciona, pero en VS Code/Jupyter se ve entrecortada y no
se puede rotar la cámara mientras se mueve.

Aquí hacemos lo mismo (la misma secuencia de configuraciones), pero:

1. Interpolamos suavemente entre cada par de configuraciones con `jtraj` (trayectoria
   polinomial de 5º orden, velocidad y aceleración cero en los extremos).
2. En vez del dibujo de "palitos" que hace `robot.plot()` de roboticstoolbox, dibujamos
   cada eslabón como un **cilindro 3D sólido** (con esferas en las juntas), usando
   `ax.plot_surface` de matplotlib sobre las mismas matrices DH manuales de la Parte 1.
   Se ve bastante más parecido a un brazo real.
3. La animación corre en la **ventana aparte** de `%matplotlib tk` (Parte 1), así que se
   puede arrastrar con el mouse para rotar la cámara incluso mientras se mueve.
4. Al final se ve el gripper **abierto**, con sus dos dedos, igual que en la referencia:
   la fila 5 de la tabla DH ($\alpha=-\pi/2$, $a=0$, $d=0$) es una rotación pura
   alrededor de la muñeca, así que los dos dedos son simplemente dos segmentos que
   salen del mismo punto (la muñeca) usando $\theta_5$ y $\pi-\theta_5$.

> Nota: existe también el backend **Swift** de roboticstoolbox (3D interactivo en el
> navegador, con sombras y controles de cámara tipo videojuego). Lo probamos, pero para
> verse bien necesita asignarle una geometría (malla o cilindro) a cada eslabón a mano —
> nuestro robot es un `DHRobot` genérico, sin mallas — así que por ahora nos quedamos con
> la versión de cilindros sólidos en matplotlib, que no necesita nada adicional.

In [ ]:
from roboticstoolbox import jtraj

# Misma secuencia del caso de estudio de referencia: parte de una postura "home"
# y mueve una articulación a la vez, terminando con theta5 (que abre/orienta el gripper).
q_home  = [0.0,  np.radians(-90), np.radians(0),   np.radians(90), np.radians(0)]
q_d1    = [-0.3, np.radians(-90), np.radians(0),   np.radians(90), np.radians(0)]
q_t2    = [-0.3, np.radians(-80), np.radians(0),   np.radians(90), np.radians(0)]
q_t3    = [-0.3, np.radians(-80), np.radians(-10), np.radians(90), np.radians(0)]
q_t4    = [-0.3, np.radians(-80), np.radians(-10), np.radians(0),  np.radians(0)]
q_final = [-0.3, np.radians(-80), np.radians(-10), np.radians(0),  np.radians(-70)]

etapas = [q_home, q_d1, q_t2, q_t3, q_t4, q_final]
pasos_por_etapa = 15

qs = np.vstack([
    jtraj(q_ini, q_fin, pasos_por_etapa).q
    for q_ini, q_fin in zip(etapas[:-1], etapas[1:])
])

print("cuadros totales de la animación:", qs.shape[0])

In [ ]:
def puntos_gripper(q, largo_dedo=0.14):
    """Puntos de los dos dedos del gripper abierto, reutilizando forward_kinematics
    de la Parte 1 (matrices DH manuales). La fila 5 de la tabla (alpha=-pi/2, a=0, d=0)
    es una rotación pura, así que ambos dedos salen del mismo punto (la muñeca) y solo
    cambian de dirección según theta5 y pi-theta5."""
    d1, th2, th3, th4, th5 = q

    frames_a = forward_kinematics(q)
    muneca = frames_a[4][:3, 3]
    dedo1 = (frames_a[5] @ np.array([largo_dedo, 0, 0, 1]))[:3]

    q_espejo = [d1, th2, th3, th4, np.pi - th5]
    frames_b = forward_kinematics(q_espejo)
    dedo2 = (frames_b[5] @ np.array([largo_dedo, 0, 0, 1]))[:3]

    return muneca, dedo1, dedo2

In [ ]:
def cilindro(ax, p0, p1, radio, color, n=10):
    """Dibuja un cilindro sólido entre los puntos 3D p0 y p1 (representa un eslabón)."""
    p0, p1 = np.array(p0), np.array(p1)
    v = p1 - p0
    mag = np.linalg.norm(v)
    if mag < 1e-9:  # p0 y p1 coinciden (p.ej. una junta que solo rota, sin desplazamiento)
        return
    v = v / mag
    no_v = np.array([1, 0, 0]) if abs(v[0]) < 0.9 else np.array([0, 1, 0])
    n1 = np.cross(v, no_v); n1 /= np.linalg.norm(n1)
    n2 = np.cross(v, n1)

    t, theta = np.meshgrid(np.linspace(0, mag, 2), np.linspace(0, 2*np.pi, n))
    X = p0[0] + v[0]*t + radio*np.sin(theta)*n1[0] + radio*np.cos(theta)*n2[0]
    Y = p0[1] + v[1]*t + radio*np.sin(theta)*n1[1] + radio*np.cos(theta)*n2[1]
    Z = p0[2] + v[2]*t + radio*np.sin(theta)*n1[2] + radio*np.cos(theta)*n2[2]
    ax.plot_surface(X, Y, Z, color=color, shade=True, linewidth=0)


def esfera(ax, centro, radio, color):
    """Dibuja una esfera sólida (representa una junta)."""
    u, v = np.meshgrid(np.linspace(0, 2*np.pi, 12), np.linspace(0, np.pi, 8))
    x = centro[0] + radio*np.cos(u)*np.sin(v)
    y = centro[1] + radio*np.sin(u)*np.sin(v)
    z = centro[2] + radio*np.cos(v)
    ax.plot_surface(x, y, z, color=color, shade=True, linewidth=0)


def dibujar_robot(ax, q):
    """Dibuja el robot completo (eslabones + juntas + gripper abierto) en la pose q."""
    frames = forward_kinematics(q)
    puntos = [F[:3, 3] for F in frames]

    colores = ['#888888', '#3b6ea5', '#3b6ea5', '#5aa469', '#c97a3d']
    radios  = [0.025,     0.02,      0.02,      0.015,     0.012]
    for i in range(len(puntos) - 1):
        cilindro(ax, puntos[i], puntos[i + 1], radios[i], colores[i])
    for p in puntos:
        esfera(ax, p, 0.028, '#333333')

    muneca, dedo1, dedo2 = puntos_gripper(q)
    cilindro(ax, muneca, dedo1, 0.008, 'blue')
    cilindro(ax, muneca, dedo2, 0.008, 'blue')

In [ ]:
# Límites fijos (a ojo) para que la ventana no se reescale sola en cada cuadro
limites = [-0.1, 0.7, -0.05, 0.5, 0, 0.6]

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, projection='3d')

for q in qs:
    ax.clear()
    dibujar_robot(ax, q)
    ax.set_xlim(limites[0], limites[1])
    ax.set_ylim(limites[2], limites[3])
    ax.set_zlim(limites[4], limites[5])
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title('Manipulador 5GDL')
    plt.pause(0.03)  # pausa corta entre cuadros -> anima suave en la ventana

plt.show()  # deja la ventana abierta para poder rotarla con el mouse al terminar

## Pose exacta del dibujo del enunciado

Con la versión "sólida" de la Parte 3 podemos comprobar que $\theta_2=-90°$ (con
$d_1=0.25$, $\theta_3=\theta_4=\theta_5=0$) reproduce fielmente la forma del diagrama:
riel, poste hacia arriba, brazo horizontal, caída a la muñeca y gripper abierto.

In [ ]:
q_imagen = [0.25, -np.pi/2, 0, 0, 0]

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, projection='3d')
dibujar_robot(ax, q_imagen)
ax.set_xlim(-0.1, 0.6); ax.set_ylim(-0.4, 0.1); ax.set_zlim(0, 0.6)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Pose del dibujo del enunciado')
plt.show()

---

# Punto 2 — Interfaz web e IoT con ESP32

*(Pendiente — se resuelve más adelante, por fuera de este notebook.)*